# Table constraining effect

In [ ]:
import numpy as np
import pandas as pd
import xarray as xr
import geopandas as gpd
import matplotlib.pyplot as plt

In [ ]:
dir01 = '../paper_ecosystem_carbon/output/01_data_prep/'
dir05 = '../paper_ecosystem_carbon/output/05_rf_constrain/'
dir07 = '../paper_ecosystem_carbon/output/07_eval_prep/'
dir08 = '../paper_ecosystem_carbon/output/08_eval/'
dir_nearth = '../data/naturalearth/'

---

In [ ]:
dict_etype = dict(trop_ef = 'Tropical/subtropical evergreen forest', 
                  trop_df = 'Tropical/subtropical deciduous forest',
                  temp_ebf = 'Temperate/boreal evergreen broadleaf forest',
                  temp_dbf = 'Temperate/boreal deciduous broadleaf forest',
                  temp_enf = 'Temperate/boreal evergreen needleleaf forest',
                  temp_dnf = 'Temperate/boreal deciduous needleleaf forest',
                  glob_shrub = 'Shrublands',
                  glob_grass = 'Grasslands',
                  glob_tundr = 'Tundra',
                  glob_mount = 'Montane')

In [ ]:
# Get predicted and constrained data
ds_rfconstr = xr.open_zarr(f'{dir05}ds_rfconstr_r025.zarr')
ds_rfpred = xr.open_mfdataset(f'{dir05}ds_rfpred_*_r025.zarr', engine='zarr')

# For maps
ds_occur = xr.open_zarr(f'{dir07}ds_prep_occur_r025.zarr')
ds_land = xr.open_dataset(f'{dir01}ds_luh2_land_mask_r025.nc')
coastline110 = gpd.read_file(
    f'{dir_nearth}ne_110m_coastline/ne_110m_coastline.shp')

---

In [ ]:
def get_constraint_stats(ctype, etype):

    """Returns constrained and unconstrained means and changes"""
    
    da_rfpred = ds_rfpred[f'{ctype}_{etype}'].where(ds_occur[etype])
    da_rfconstr = ds_rfconstr[f'{ctype}_{etype}'].where(ds_occur[etype])

    unconstr_mean = int(round(da_rfpred.mean().compute().item(), 0))
    constr_mean = int(round(da_rfconstr.mean().compute().item(), 0))

    abs_change = constr_mean - unconstr_mean

    if np.isclose(unconstr_mean, 0):
        rel_change = np.nan
    else:
        rel_change = round((100 * abs_change / unconstr_mean), 1)

    return {
        'Unconstrained mean': unconstr_mean,
        'Constrained mean': constr_mean,
        'Absolute change': abs_change,
        'Relative change (%)': rel_change,
    }


def make_constraint_table(ctype):

    """Returns table with constrained and unconstrained stats"""
    
    return pd.DataFrame([
        {
            'Type': etype_name,
            **get_constraint_stats(ctype, etype),
        }
        for etype, etype_name in dict_etype.items()
    ]).set_index('Type')


# Run for cveg and csoil
dfx_veg = make_constraint_table('cveg')
dfx_soil = make_constraint_table('csoil')

# Export
dfx_veg.to_csv(f'{dir08}tables/table_effect_constrain_cveg.csv', 
               mode='w', index=True)
dfx_soil.to_csv(f'{dir08}tables/table_effect_constrain_csoil.csv', 
               mode='w', index=True)

In [ ]:
dfx_veg

In [ ]:
dfx_soil